# Notebook 02 — LLMs as Research Instruments



**Duration:** ~20 minutes



In notebook 01 you connected to a large language model (LLM) and asked it a question. In this notebook we look at the model more carefully: what does it already know, where does its knowledge break down, and why does it sometimes give different answers to the same question?



An LLM is a system trained on a very large amount of text. It learns patterns from that text — facts, grammar, reasoning styles — and uses those patterns to generate new text. It does not look anything up. It does not search the internet. Everything it produces comes from patterns it absorbed during training.



This makes it useful, but also unreliable in specific ways. By the end of this notebook you will understand those limitations well enough to work with them.

## Setup

Run this cell first. If you stored your Groq API key in Colab Secrets (notebook 01), it will load automatically.

In [ ]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================

!pip install groq requests lxml Pillow
import os, json, base64, requests, io
from groq import Groq
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

# Load API key from Colab Secrets (set up in notebook 01)
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    os.environ["GROQ_API_KEY"] = "paste_your_key_here"   # <-- fallback: paste key here
    print("Could not load from Secrets. Paste your key in the line above.")

client = Groq(api_key=os.environ["GROQ_API_KEY"])
TEXT_MODEL = "llama-3.3-70b-versatile"
VISION_MODEL = "meta-llama/llama-4-maverick-17b-128e-instruct"

print("Setup complete.")

## What does the model already know?

The model was trained on a broad range of text — books, websites, legal documents, academic papers. That means it has **implicit knowledge**: things it absorbed during training, without being given a specific source to read.

This is useful. But it is also risky, because you cannot tell which facts it learned accurately and which ones it reconstructed imperfectly.

Let's test this. We will ask the model about two real New Zealand Acts — without giving it any source text to read.

## Exercise a — A well-known Act

The Privacy Act 2020 is widely discussed in legal and policy writing. The model has almost certainly seen many documents about it during training.

Run the cell below. We are not giving the model any data — just asking what it already knows.

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    messages=[
        {"role": "user", "content": "What is the purpose of New Zealand's Privacy Act 2020?"}
    ]
)

print(response.choices[0].message.content)

You should see a confident, detailed answer. The model sounds like it knows this Act well.

Now let's try something less well-known.

## Exercise b — An obscure Act

The Impounding Act 1955 is a real New Zealand Act that is still in force. It deals with a very specific (and quite old-fashioned) area of law. It is unlikely to appear in many modern training documents.

We will ask the model two questions about it — without giving it any source text. See how it goes.

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    messages=[
        {"role": "user", "content": "What is the main purpose of New Zealand's Impoundment Act 1955?"}
    ]
)

print(response.choices[0].message.content)

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    messages=[
        {"role": "user", "content": "New Zealand's Impoundment Act 1955, what is the maximum fine for not discosing the destruction of wandering livestock onto your property given in `Special remedies for trespass by pigs, goats, or poultry`?"}
    ]
)

print(response.choices[0].message.content)

Act sources:
- Impoundment ACT: https://www.legislation.govt.nz/act/public/1955/108/en/latest/#DLM293864
- Privacy Act: https://www.legislation.govt.nz/act/public/2020/0031/latest/LMS23223.html

Compare the two answers:

1. Did the model sound equally confident about both Acts?
2. Did it give specific, verifiable details about the Impoundment Act, or did it produce vague, plausible-sounding text?
3. Would you trust either answer without checking the source?

"Hallucinations"
What we've seen is **implicit knowledge** in action. The model does not tell you when it is guessing. It produces fluent text regardless of whether its information is accurate. For well-documented topics it is often right. For obscure ones it may **hallucinate** — generate confident-sounding text that is partially or entirely wrong.

This is why, in notebooks 03–05, we will always give the model **real source data** to work with rather than relying on what it already knows.

## Consistency and temperature

Why do we have different answers to the same question? The model is not a calculator that gives the same output for the same input. It is a text generator that produces different outputs each time, based on **probabilities**.

DIfferent results each run can be an issue for research - which seeks the opposite.

A setting called **temperature** controls how varied LLM answers are:

| Temperature | Behaviour | Use case |
|-------------|-----------|----------|
| `0` | Always picks the most probable word. The output is nearly identical every time. | Research tasks where consistency matters |
| `1` | Picks words with more randomness. The output varies between runs. | Wider range of topics / keyword searches |

## Exercise c — Temperature comparison

We will send the **exact same prompt** to the model four times:
- Twice at `temperature=0` (always the same)
- Twice at `temperature=1` (more random)

Watch for differences between runs.

### Temperature = 0 (first run)

In [ ]:
prompt = "From the Impoundment Act 1955, list the Special remedies for trespass by pigs, goats, or poultry. Be consise."

response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("=== Temperature 0, Run 1 ===")
print(response.choices[0].message.content)

### Temperature = 0 (second run)

Run this cell. The prompt is identical. Is the answer the same?

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=0,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("=== Temperature 0, Run 2 ===")
print(response.choices[0].message.content)

The two answers should be identical or very close. Now let's see what happens when we increase the temperature.

### Temperature = 1 (first run)

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=1,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("=== Temperature 1, Run 1 ===")
print(response.choices[0].message.content)

### Temperature = 1 (second run)

Same prompt again. Compare with the run above.

In [ ]:
response = client.chat.completions.create(
    model=TEXT_MODEL,
    temperature=1,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print("=== Temperature 1, Run 2 ===")
print(response.choices[0].message.content)

### What did you notice?

At temperature 0, the two answers should be the same (or nearly). At temperature 1, they probably differ — different wording, different examples, maybe different obligations entirely.

Neither answer is wrong. But they are **inconsistent**. If you used this to classify 200 legislative sections, different runs might produce different classifications for the same section.

## Exercise d — Inconsistency versus creativity

So far we have treated `temperature=1` as a problem — a source of inconsistency to be avoided. But that is not the whole story. Higher temperature makes the model produce a wider spread of answers. For some research tasks, that spread is exactly what you want.

Imagine you have a passage of text and you want to come up with keywords to code it. A single run at `temperature=0` gives you one consistent answer — good for reproducibility, but you might miss useful alternatives. Running at `temperature=1` gives you a different menu of candidate keywords to consider.

**First, you try.** Read the passage below and pick **10 keywords** that you think capture the substantive concepts. Then we will ask the model to do the same at `temperature=0` and `temperature=1`, and look at the three lists side by side.

In [ ]:
PRIVACY_ACT_PASSAGE = """
What is personal information and the Privacy Act? 

Personal information is “information about an identifiable individual”. It covers both information that is simply about a person (e.g. eye colour) and information that may also identify them (e.g. their name). The information does not need to name the individual, as long as they are identifiable in other ways, like through their home address.

The Privacy Act 2020 provides the rules in New Zealand for protecting personal information and puts responsibilities on agencies and organizations about how they must do that. For example, people have a right to know what information your agency holds about them and a right to ask you to correct it if they think it is wrong.
"""

print(PRIVACY_ACT_PASSAGE)

# Your turn — replace these with 10 keywords you pick from the passage.
my_keywords = [
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
    "____",
]

In [ ]:
keyword_prompt = (
    "Extract 10 keywords that capture the substantive concepts in this passage. "
    "Return ONLY a JSON array of lowercase strings, e.g. [\"consent\", \"disclosure\"]. "
    "No preamble. No code fences.\n\n"
    f"Passage:\n{PRIVACY_ACT_PASSAGE}"
)

def extract_keywords(temperature):
    response = client.chat.completions.create(
        model=TEXT_MODEL,
        temperature=temperature,
        messages=[{"role": "user", "content": keyword_prompt}]
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = "\n".join(l for l in raw.splitlines() if not l.startswith("```")).strip()
    return [k.lower().strip() for k in json.loads(raw)]

temp0_keywords = extract_keywords(0)
temp1_keywords = extract_keywords(1)

print("Temperature = 0:", temp0_keywords)
print()
print("Temperature = 1:", temp1_keywords)

In [ ]:
# Side-by-side comparison of the three lists.
rows = max(len(my_keywords), len(temp0_keywords), len(temp1_keywords))

header = f"{'Your keywords':<25} {'Temp = 0':<25} {'Temp = 1':<25}"
print(header)
print("-" * len(header))
for i in range(rows):
    a = my_keywords[i]    if i < len(my_keywords)    else ""
    b = temp0_keywords[i] if i < len(temp0_keywords) else ""
    c = temp1_keywords[i] if i < len(temp1_keywords) else ""
    print(f"{a:<25} {b:<25} {c:<25}")

### When is higher temperature actually useful?

Compare the three lists. Your own picks, `temperature=0`, and `temperature=1` will overlap in places and diverge in others:

- At `temperature=0`, you get one consistent list. Great for reproducibility — run it again and you get the same keywords.
- At `temperature=1`, you get a different spread. Some framings show up here that `temperature=0` would never surface.

This flips the usual advice. For **coding and extraction** — where consistency is everything — stick with `temperature=0`. But for **exploration** — when you want a menu of possible framings, keywords, or interpretations to consider — running at `temperature=1` a few times can surface ideas a single low-temperature run would miss.

## What you accomplished

In this notebook you explored two factors that shape the quality and direction of LLM outputs:

| Factor | What you observed | Exercise |
|--------|-------------------|----------|
| **Training data** | The model was confident on the well-known Privacy Act<br>but unreliable on the obscure Impounding Act.<br>Its accuracy depends on how much relevant,<br>recent data it was trained on. | a & b |
| **Temperature** | Low temperature gives reproducible answers (exercise c).<br>Higher temperature surfaces a wider spread of candidate<br>framings and keywords (exercise d).| c & d |

The key insight: an LLM is a **research instrument**, not an oracle. Like any instrument, it needs to be calibrated (temperature) and pointed at real data (not just its implicit knowledge). When either of these factors is weak, the output suffers — and the model will not tell you.

This is why, in notebooks 03–05, we default to `temperature=0` for coding and extraction, and reach for higher temperature only when we are still deciding what to look for.

**Next up:** After the break, notebook 03 turns to prompt engineering. You will build the same request five ways, see how each layer of prompt structure changes the output, and then replicate a published coding method on a passage of real NZ legislation.